# Session 04: Debugging Lab, Imports and Tests

**Notebook 3 of 3: Debugging lab** · used in the live session's debugging segment, and for practice afterwards (about 45 minutes)

Five small projects, each broken in a way you'll meet in real Python codebases: three import problems and two test problems. For each one:

1. **Observe:** run the broken project and read the output.
2. **Hypothesize:** write down what you think is wrong *before* opening a hint.
3. **Inspect:** use the hints if you need them.
4. **Fix:** edit the copy of the files in the fix cell.
5. **Verify:** run the fix cell, and check that you get the expected result.

Every project is written to a temporary folder and run with this kernel's Python, so nothing touches your own files. Run the next cell first.

In [ ]:
import os
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path


def run_project(files: dict[str, str], *commands: list[str], src_on_path: bool = False) -> None:
    """Write files to a fresh temporary folder, then run each command with this kernel's python."""
    env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
    env |= {"PYTHONDONTWRITEBYTECODE": "1", "NO_COLOR": "1", "PY_COLORS": "0"}  # plain text output
    with tempfile.TemporaryDirectory() as workdir:
        root = Path(workdir)
        if src_on_path:
            env["PYTHONPATH"] = str(root / "src")
        for relative_path, text in files.items():
            path = root / relative_path
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")
        for command in commands:
            print(f"$ python {' '.join(command)}")
            result = subprocess.run(
                [sys.executable, *command],
                cwd=root,
                capture_output=True,
                text=True,
                encoding="utf-8",
                env=env,
            )
            output = (result.stdout + result.stderr).rstrip()
            print(output.replace(str(root), "<project>"))
            print(f"[exit code {result.returncode}]\n")


print("run_project() is ready")

## Lab 1: "But `json` is in the standard library!"

A teammate added a small helper module to the nightly job's folder. Since then, the job fails on its very first line of real work.

In [ ]:
LAB1 = {
    "json.py": """
        \"\"\"Helpers for printing JSON nicely in reports.\"\"\"


        def pretty(data):
            return str(data)
        """,
    "nightly.py": """
        import json

        report = {"open": 9, "breached": 6}
        print(json.dumps(report))
        """,
}

run_project(LAB1, ["nightly.py"])

**Observe:** read the **whole** last line of the traceback. The `json` module has no `dumps`? It has had one since Python 2.6. Python 3.13 adds a hint in brackets at the end; older versions don't, so make sure you can explain the cause without it.

<details>
<summary>Hint 1</summary>

**Inspect** which file Python actually imported. Add `print(json.__file__)` to `nightly.py`, straight after the import, and run it again.

</details>

<details>
<summary>Hint 2: the explanation</summary>

`sys.path` starts with the script's own folder (see the import diagram in `01-concepts.ipynb`), so `import json` found the teammate's `json.py` before the standard library's. This is **shadowing**. It happens with any standard-library name: `logging.py`, `random.py`, `email.py`, `test.py`, `code.py`...

**Fix:** rename the helper to something distinctive, such as `json_helpers.py`, and update anything that imported it. Also delete any `__pycache__/json*.pyc` left behind in a real project.

</details>

Write your fix by editing `LAB1_FIXED` below, then run the cell. **Verify:** it prints `{"open": 9, "breached": 6}` with exit code `0`.

In [ ]:
LAB1_FIXED = dict(LAB1)   # start from the broken files; change keys or contents below
# For example: LAB1_FIXED["new_name.py"] = LAB1_FIXED.pop("json.py")

run_project(LAB1_FIXED, ["nightly.py"])

## Lab 2: The module that only works sometimes

A package in the `src/` layout. Its author runs the CLI module directly as a file, and gets one error. A colleague switches to a relative import, and gets a different one.

In [ ]:
LAB2 = {
    "src/ops_tools/__init__.py": "",
    "src/ops_tools/greeting.py": """
        def greet(team):
            return f"Good morning, {team} on-call"
        """,
    "src/ops_tools/cli.py": """
        from ops_tools.greeting import greet

        print(greet("platform"))
        """,
    "src/ops_tools/cli_relative.py": """
        from .greeting import greet

        print(greet("network"))
        """,
}

run_project(LAB2, ["src/ops_tools/cli.py"], ["src/ops_tools/cli_relative.py"])

**Observe:** two different errors from essentially the same code. Write down your hypothesis for each before opening the hints.

<details>
<summary>Hint 1: the first error</summary>

When you run `python src/ops_tools/cli.py`, the folder added to `sys.path` is the **script's** folder, `src/ops_tools/`. The package `ops_tools` lives in `src/`, which isn't on the path, so `import ops_tools...` can't find it.

</details>

<details>
<summary>Hint 2: the second error</summary>

A relative import (`from .greeting`) means "from my own package". A file run as a script has no package: it's just `__main__`. So Python can't tell what `.` refers to.

</details>

<details>
<summary>Hint 3: the fix</summary>

Don't run files inside a package as scripts. Run them **as modules of their package**, with the package importable:

- in a uv project: `uv run python -m ops_tools.cli`, or the command defined in `[project.scripts]`;
- here, `src_on_path=True` plays the role of the editable install, and the command is `["-m", "ops_tools.cli"]`.

With `-m`, both the absolute and the relative versions work. The course uses absolute imports anyway (Part 2 of the concepts notebook).

</details>

**Verify:** change the commands below so that both modules print their greeting with exit code `0`. Don't change the files.

In [ ]:
run_project(LAB2, ["src/ops_tools/cli.py"], ["src/ops_tools/cli_relative.py"], src_on_path=False)

## Lab 3: The circular import

Two modules in a `tracker` package. `models` uses a formatting helper; the helper's author imported `Ticket` for a type hint.

In [ ]:
LAB3 = {
    "src/tracker/__init__.py": "",
    "src/tracker/models.py": """
        from dataclasses import dataclass

        from tracker.formatting import short_label


        @dataclass(frozen=True)
        class Ticket:
            id: str
            team: str
            priority: str

            def label(self):
                return short_label(self)
        """,
    "src/tracker/formatting.py": """
        from tracker.models import Ticket


        def short_label(ticket: Ticket) -> str:
            return f"{ticket.id} [{ticket.priority}] {ticket.team}"
        """,
    "main.py": """
        from tracker.models import Ticket

        print(Ticket("INC-3001", "platform", "P1").label())
        """,
}

run_project(LAB3, ["main.py"], src_on_path=True)

**Observe:** Python's message already suggests the cause. Trace the order of events yourself: which module starts loading first, which line triggers the second module, and what the second module finds when it looks inside the first.

<details>
<summary>Hint 1: the sequence</summary>

1. `main.py` imports `tracker.models`. Python creates an empty module object, stores it in `sys.modules`, and starts running `models.py` from the top.
2. Line 3 of `models.py` imports `tracker.formatting`, which starts running.
3. `formatting.py` asks for `Ticket` from `tracker.models`. That module is in `sys.modules`, but only **partially initialised**: execution is still on line 3, and `class Ticket` hasn't been defined yet. So the name doesn't exist.

</details>

<details>
<summary>Hint 2: fixes, best first</summary>

- **Make the dependency point one way.** `formatting` only needs *something with* `.id`, `.priority` and `.team`, so it doesn't have to import `Ticket` at all. Duck typing makes the import unnecessary. Remove it and drop the annotation (or annotate with `object`).
- Or move `short_label` into `models.py`, since it's about tickets anyway.
- If a type hint is genuinely needed, the standard pattern is `from typing import TYPE_CHECKING` with the import inside `if TYPE_CHECKING:`, which runs only for type checkers. Use it sparingly: it treats the symptom.

</details>

**Verify:** edit `LAB3_FIXED` so that it prints `INC-3001 [P1] platform` with exit code `0`.

In [ ]:
LAB3_FIXED = dict(LAB3)

run_project(LAB3_FIXED, ["main.py"], src_on_path=True)

## Lab 4: Green tests, wrong code

`open_count` has a bug, yet both tests pass. Run them, then look closely at the test file and the output.

In [ ]:
LAB4 = {
    "summary.py": """
        def open_count(tickets):
            return sum(1 for ticket in tickets if ticket["status"] == "open")
        """,
    "test_summary.py": """
        from summary import open_count

        TICKETS = [
            {"id": "INC-1", "status": "open"},
            {"id": "INC-2", "status": "in_progress"},
            {"id": "INC-3", "status": "closed"},
        ]


        def test_in_progress_counts_as_open():
            assert (open_count(TICKETS) == 2, "in_progress tickets are open too")


        def test_empty_list():
            open_count([])
        """,
}

run_project(LAB4, ["-m", "pytest", "-q", "-p", "no:cacheprovider"])

**Observe:** "2 passed", plus a warning. The business rule says `in_progress` counts as open, so `open_count(TICKETS)` should be `2`, but the code returns `1`. How did the test pass?

<details>
<summary>Hint 1: the first test</summary>

`assert (condition, "message")` asserts a **tuple**. A non-empty tuple is always truthy (Session 02), so the assertion can never fail. pytest's warning says exactly that: "assertion is always true, perhaps remove parentheses?". Write `assert condition, "message"` instead. And treat warnings in test output as bugs.

</details>

<details>
<summary>Hint 2: the second test</summary>

It calls the function but asserts nothing, so it only proves the function doesn't crash. Add `assert open_count([]) == 0`.

</details>

**Fix** the **tests** first, and **verify** that `test_in_progress_counts_as_open` now **fails**: a good test must fail on buggy code. Then fix `summary.py` (count everything that isn't `"resolved"` or `"closed"`), and verify that both tests pass with no warnings.

In [ ]:
LAB4_FIXED = dict(LAB4)

run_project(LAB4_FIXED, ["-m", "pytest", "-q", "-p", "no:cacheprovider"])

## Lab 5: Passes alone, fails together

Each test passes when run on its own. Run the whole file, and one fails.

In [ ]:
LAB5 = {
    "triage.py": """
        def close_resolved(tickets):
            \"\"\"Mark every 'resolved' ticket as 'closed'. Returns how many changed.\"\"\"
            changed = 0
            for ticket in tickets:
                if ticket["status"] == "resolved":
                    ticket["status"] = "closed"
                    changed += 1
            return changed
        """,
    "test_triage.py": """
        from triage import close_resolved

        TICKETS = [
            {"id": "INC-1", "status": "open"},
            {"id": "INC-2", "status": "resolved"},
        ]


        def test_closes_resolved_tickets():
            assert close_resolved(TICKETS) == 1


        def test_open_tickets_are_untouched():
            assert close_resolved(TICKETS) == 1
            assert TICKETS[0]["status"] == "open"
        """,
}

ONE_TEST = "test_triage.py::test_open_tickets_are_untouched"

run_project(
    LAB5,
    ["-m", "pytest", "-q", "-p", "no:cacheprovider", ONE_TEST],   # this test on its own
    ["-m", "pytest", "-q", "-p", "no:cacheprovider"],             # the whole file
)

**Observe:** run alone, `test_open_tickets_are_untouched` passes. Run after `test_closes_resolved_tickets`, it fails, and the failure says `close_resolved` returned `0`.

<details>
<summary>Hint 1: what changed between the runs?</summary>

`TICKETS` is one module-level list of dicts, shared by every test in the file. `close_resolved` mutates the dicts **in place** (Session 02's aliasing), so after the first test runs, `INC-2` is already `"closed"`, and the second test finds nothing left to close. The tests depend on the order they run in, which is the worst kind of flaky.

</details>

<details>
<summary>Hint 2: the fix</summary>

Give each test its own fresh data with a fixture:

```python
import pytest

@pytest.fixture
def tickets():
    return [
        {"id": "INC-1", "status": "open"},
        {"id": "INC-2", "status": "resolved"},
    ]

def test_closes_resolved_tickets(tickets):
    assert close_resolved(tickets) == 1
```

pytest calls the fixture **once per test**, so each test gets a new list. (A frozen dataclass would have prevented the mutation in the first place: compare Part 4 of the concepts notebook.)

</details>

**Verify:** after your fix, the full run reports `2 passed`.

In [ ]:
LAB5_FIXED = dict(LAB5)

run_project(LAB5_FIXED, ["-m", "pytest", "-q", "-p", "no:cacheprovider"])

## Reflection

- Which of these five would have taken you longest to find without the hints, and what was the first clue in the output?
- Labs 1–3 all come from **where Python looks** and **in what order things run**. Labs 4–5 come from tests that check less than they appear to. Which kind is more dangerous in production, and why?

Next: the self-check quiz in `02-practice-quiz.ipynb`, then the workbook.